### Biblioteki

In [ ]:
import numpy as np
import pandas as pd
import plotly as plt
import dash
import json

### Import

In [ ]:

with open(r'Dane/PC-AFK-K-03.01.2026-.json', 'r', encoding='utf-8') as f: #Scieżka
    raw_data = json.load(f)

bucket_name = "aw-watcher-afk_Pc-Gaming-RTX" #Nazwa bucketa w której jest wewnętrzna nazwa urządzenia
events_list = raw_data['buckets'][bucket_name]['events']

df1 = pd.json_normalize(events_list)
# df1['timestamp'] = pd.to_datetime(df1['timestamp'])

with open(r'Dane/PC-APP-K-03.01.2026-.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

bucket_name = "aw-watcher-window_Pc-Gaming-RTX"
events_list = raw_data['buckets'][bucket_name]['events']

df2 = pd.json_normalize(events_list)


### Ultimate APP + AFK

In [ ]:
import pandas as pd
import numpy as np

# Zakładam, że df1 (AFK) i df2 (Window) są już wczytane
df_afk = df1.copy()
df_window = df2.copy()

# --- KROK 1: Konwersja i przygotowanie ---
# Konwersja na datetime (UTC) - to jest "normalny, wydajny format"
df_afk['timestamp'] = pd.to_datetime(df_afk['timestamp'], format='ISO8601', utc=True)
df_window['timestamp'] = pd.to_datetime(df_window['timestamp'], format='ISO8601', utc=True)

# Opcjonalnie: Konwersja na strefę czasową Polski (dla czytelności)
# df_afk['timestamp'] = df_afk['timestamp'].dt.tz_convert('Europe/Warsaw')
# df_window['timestamp'] = df_window['timestamp'].dt.tz_convert('Europe/Warsaw')

# Obliczenie czasu zakończenia (Timedelta jest bardzo wydajna)
df_afk['end_time'] = df_afk['timestamp'] + pd.to_timedelta(df_afk['duration'], unit='s')
df_window['end_time'] = df_window['timestamp'] + pd.to_timedelta(df_window['duration'], unit='s')

# --- KROK 2: Oś czasu (Master Timeline) ---
all_timestamps = pd.concat([
    df_window['timestamp'], 
    df_window['end_time'], 
    df_afk['timestamp'], 
    df_afk['end_time']
]).unique()

# Sortujemy i tworzymy DataFrame (tylko RAZ, usuwając duplikat z Twojego kodu)
df_master = pd.DataFrame({'timestamp': all_timestamps})
df_master = df_master.sort_values('timestamp').reset_index(drop=True)

# --- KROK 3: Łączenie (Merge Asof) ---
# Przygotowanie nazw kolumn
df_window = df_window.rename(columns={'data.app': 'app', 'data.title': 'title'})
df_afk = df_afk.rename(columns={'data.status': 'status'})

# Dołączanie informacji o Oknie
df_master = pd.merge_asof(
    df_master, 
    df_window[['timestamp', 'end_time', 'app', 'title']].sort_values("timestamp"), 
    on='timestamp', 
    direction='backward',
    suffixes=('', '_win')
)

# Dołączanie informacji o AFK
df_master = pd.merge_asof(
    df_master, 
    df_afk[['timestamp', 'end_time', 'status']].sort_values("timestamp"), 
    on='timestamp', 
    direction='backward',
    suffixes=('', '_afk')
)

# --- KROK 4: Obliczenie Duration (NAJWAŻNIEJSZA ZMIANA) ---
# Liczymy czas trwania TERAZ, na pełnej osi czasu, włączając przerwy
# Dzięki temu "dziura" w czasie zostanie policzona jako osobny (później odrzucony) wiersz
df_master['duration'] = df_master['timestamp'].diff().shift(-1).dt.total_seconds()

# --- KROK 5: Filtrowanie aktywności ---
is_window_active = df_master['timestamp'] < df_master['end_time']
is_afk_active = df_master['timestamp'] < df_master['end_time_afk']

# Zostawiamy tylko momenty, gdzie okno było faktycznie otwarte
df_final = df_master[is_window_active].copy()

# Uzupełnianie statusu AFK
# Jeśli brak danych o AFK, a okno działa, zakładamy 'not-afk'
# Używamy np.where dla wydajności wektorowej (szybsze niż .loc przy dużych danych)
df_final['status'] = np.where(
    df_final['timestamp'] < df_final['end_time_afk'], 
    df_final['status'], 
    'not-afk'
)

# --- KROK 6: Czyszczenie końcowe ---
# Usuwamy wiersze z zerowym czasem lub błędami
df_final = df_final[df_final['duration'] > 0]

# Wybór kolumn
df = df_final[['timestamp', 'duration', 'app', 'title', 'status']].reset_index(drop=True)

# --- SPRAWDZENIE FORMATU ---
print("Typy danych:")
print(df.dtypes)
print("\nPrzykładowe dane:")
print(df.head())

In [ ]:
df

In [ ]:
wynik = df[df['status'] == 'not-afk'].groupby('app')['duration'].sum()
wynik.sort_values(ascending=False)

In [ ]:
wynik = df[df['status'] == 'afk'].groupby('app')['duration'].sum()
wynik.sort_values(ascending=False)

In [ ]:
print(df2.groupby("data.app")["duration"].sum().sort_values(ascending=False))


In [ ]:
df.to_json('Dane/PC-SUM-K-03.01.2026.json', orient='records', indent=4, date_format='iso',force_ascii=False)